# Experimental Data Processing Example

This notebook demonstrates how we process raw plate-reader data into a clean, filtered dataset suitable for training surrogate models. It walks through a single round of a PETase optimization campaign as a concrete example.

## Overview

Our assay measures **enzyme activity** (PETase → 4-MU fluorescence kinetics) and **expression level** (mScarlett fluorescence endpoint) in 384-well plates. Each variant is plated in triplicate across the plate, and we read the same plate at multiple time points to build a kinetics curve.

**Processing pipeline:**
1. **Parse** raw instrument files and plate maps
2. **Apply standard curves** to convert fluorescence → concentration
3. **Fit kinetics** — linear regression on 4-MU concentration vs. time
4. **Normalize** — subtract background, divide by expression (mScarlett)
5. **Aggregate** replicates and apply QC filters
6. **Normalize to positive control** to get activity relative to reference

## Input files

| File | Description |
|---|---|
| `input/raw_data/*.txt` | BioTek Neo2 exports — each file is one time point containing an mScarlett endpoint read and a 4-MU kinetics read |
| `input/plate_maps/*.csv` | Maps each well to a variant name, sequence, sample type (sample / positive_control / negative_control), and replicate number |
| `input/standard_curves.json` | Linear standard-curve parameters (slope, intercept) for each instrument, used to convert fluorescence to µM |
| `input/fastas/*.fasta` | Protein sequences for the variants in this round |

In [ ]:
import sys
import os
import glob
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn.metrics
from tqdm.auto import tqdm
import warnings

warnings.simplefilter(action="ignore", category=pd.errors.SettingWithCopyWarning)
sns.set_theme(style="whitegrid")
%matplotlib inline

# Import utility functions from the example_data folder
EXAMPLE_DIR = Path("../example_data/data_processing").resolve()
sys.path.insert(0, str(EXAMPLE_DIR))
from utils import get_mscarlett_and_kinetics, get_least_squares_fit

## Configuration

These parameters control how data is parsed and filtered. Adjust them based on your assay:

- **`mscarlett_threshold`** — minimum mScarlett concentration (µM) to consider a well as expressing. Wells below this are likely empty or failed transformations.
- **`cv_threshold`** — maximum coefficient of variation across replicates. Variants with high CV have inconsistent replicates and should be flagged.
- **`rounds_to_process`** — which experimental rounds to include.
- **`timepoints_to_process`** — which time-point files to use for kinetics fitting (we typically skip t0 since the substrate hasn't had time to react).

In [ ]:
folder_to_process = EXAMPLE_DIR
standard_curve_path = str(EXAMPLE_DIR / "input" / "standard_curves.json")

mscarlett_threshold = 0.1   # µM — wells below this are considered non-expressing
cv_threshold = 1.0          # max replicate CV for QC filtering
mscarlett_num_pass_threshold = 3  # require at least this many replicates passing mScarlett threshold

rounds_to_process = [4]
timepoints_to_process = [1, 2]

print(f"Example data directory: {EXAMPLE_DIR}")
print(f"Raw data files: {sorted(glob.glob(str(EXAMPLE_DIR / 'input' / 'raw_data' / '*.txt')))}")

In [ ]:
def get_next_int(string: str, split_str: str) -> int:
    """Extract the integer immediately following a keyword in a string.

    For example, get_next_int('round4_sample0_plate3', 'round') returns 4.
    """
    parts = string.split(split_str)
    assert len(parts) == 2, f"Expected one occurrence of '{split_str}' in '{string}'"
    digit_str = ""
    for char in parts[1]:
        if char.isdigit():
            digit_str += char
        else:
            break
    return int(digit_str)

## Step 1: Parse Raw Data

For each time point, we:
1. Find the matching raw data file(s) and plate map(s)
2. Parse both the **mScarlett endpoint** (expression proxy) and **4-MU kinetics** (enzyme activity) from the same instrument file
3. Apply standard curves to convert fluorescence → micromolar concentration
4. Merge with plate-map metadata

After parsing, each time point's kinetics data is compressed to one row per well by averaging the 4-MU concentrations across the kinetics reads within that time point. This gives us one 4-MU measurement per (well, time point) pair that we can use for fitting.

In [ ]:
mscarlett_df_list = []
kinetics_df_list = []

for round_number in rounds_to_process:
    for i in timepoints_to_process:
        to_process = glob.glob(
            str(folder_to_process / "input" / "raw_data" / f"*round{round_number}*t{i}*.txt")
        )
        if not to_process:
            continue

        mapping_files = glob.glob(
            str(folder_to_process / "input" / "plate_maps" / "*.csv")
        )

        mapping_list = []
        for p in to_process:
            rn = get_next_int(p, "round")
            sn = get_next_int(p, "sample")
            pn = get_next_int(p, "plate")
            matches = [
                m for m in mapping_files
                if f"round{rn}" in m and f"sample{sn}" in m and f"plate{pn}" in m
            ]
            assert len(matches) == 1, f"Expected one mapping file for {p}, got: {matches}"
            mapping_list.append(matches[0])

        mscarlett_df, kinetics_df = get_mscarlett_and_kinetics(
            mapping_list, to_process, standard_curve_path
        )

        # Compress kinetics: average 4-MU across reads within each time point
        compressed_list = []
        for sample in kinetics_df["sample_number"].unique():
            tmp_sample = kinetics_df[kinetics_df["sample_number"] == sample]
            for plate in tmp_sample["plate_number"].unique():
                tmp_plate = tmp_sample[tmp_sample["plate_number"] == plate]
                for well in tmp_plate["well"].unique():
                    well_data = tmp_plate[tmp_plate["well"] == well].copy()
                    well_data["4MU_um_mean"] = well_data["4MU_um"].mean()
                    compressed_list.append(
                        well_data.drop(columns=["time", "value", "4MU_um"]).head(1)
                    )

        kinetics_df = pd.concat(compressed_list).reset_index(drop=True)
        kinetics_df["timepoint"] = i
        mscarlett_df["timepoint"] = i

        mscarlett_df_list.append(mscarlett_df)
        kinetics_df_list.append(kinetics_df)

mscarlett_df = pd.concat(mscarlett_df_list).reset_index(drop=True)
kinetics_df = pd.concat(kinetics_df_list).reset_index(drop=True)

print(f"Parsed {len(mscarlett_df)} mScarlett readings and {len(kinetics_df)} kinetics readings")
print(f"Unique wells: {kinetics_df['well'].nunique()}")
print(f"Time points used: {sorted(kinetics_df['timepoint'].unique())}")

## Step 2: Fit Kinetics

For each well, we fit a line through the (timestamp, 4-MU concentration) points across time points. The **slope** is the raw reaction rate — how fast the enzyme converts substrate to product.

We also record the **R² score** to flag wells where the kinetics are non-linear (e.g., substrate depletion, enzyme inactivation). Wells with low R² should be treated with caution.

The mScarlett reading is averaged across time points to get a single expression estimate per well.

In [ ]:
df_kinetic_fit_list = []

for round_number in kinetics_df["round_number"].unique():
    tmp_round = kinetics_df[kinetics_df["round_number"] == round_number]
    for sample in tmp_round["sample_number"].unique():
        tmp_sample = tmp_round[tmp_round["sample_number"] == sample]
        for plate in tmp_sample["plate_number"].unique():
            tmp_kinetics = tmp_sample[tmp_sample["plate_number"] == plate]
            tmp_mscarlett = mscarlett_df[
                (mscarlett_df["plate_number"] == plate)
                & (mscarlett_df["sample_number"] == sample)
                & (mscarlett_df["round_number"] == round_number)
            ]

            fit_list = []
            for well in tmp_kinetics["well"].unique():
                well_kin = tmp_kinetics[tmp_kinetics["well"] == well].sort_values("timepoint")
                well_msc = tmp_mscarlett[tmp_mscarlett["well"] == well].sort_values("timepoint")

                xs = well_kin["datetime"].values
                ys = well_kin["4MU_um_mean"].values

                slope, intercept = get_least_squares_fit(xs, ys)
                r2 = sklearn.metrics.r2_score(ys, slope * xs + intercept)

                row = well_kin.drop(columns=["datetime", "timepoint", "4MU_um_mean", "source_file"]).head(1).copy()
                row["raw_rate"] = slope
                row["r2_score"] = r2
                row["mscarlett_um_mean"] = well_msc["mscarlett_um"].mean()
                fit_list.append(row)

            df_kinetic_fit_list.append(pd.concat(fit_list).reset_index(drop=True))

kinetics_fit_df = pd.concat(df_kinetic_fit_list).reset_index(drop=True)

print(f"Fitted {len(kinetics_fit_df)} wells")
print(f"R² distribution:")
print(kinetics_fit_df["r2_score"].describe().to_string())

### QC: Negative Control Kinetics

Negative controls (no enzyme) should show flat kinetics (near-zero slope). Plotting their fitted rates helps verify the background subtraction will be meaningful. If negative controls show high rates, there may be contamination or assay issues.

In [ ]:
nc_data = kinetics_fit_df[kinetics_fit_df["sample_type"] == "negative_control"]

fig, ax = plt.subplots(figsize=(6, 4), dpi=100)
ax.bar(range(len(nc_data)), nc_data["raw_rate"].values, color="steelblue", alpha=0.7)
ax.set_xlabel("Negative Control Well")
ax.set_ylabel("Raw Rate")
ax.set_title(f"Negative Control Raw Rates (n={len(nc_data)})")
ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")
plt.tight_layout()
plt.show()

print(f"Mean background rate: {nc_data['raw_rate'].mean():.6f}")
print(f"Std background rate:  {nc_data['raw_rate'].std():.6f}")

## Step 3: Aggregate, Normalize, and Filter

Now we aggregate across replicates for each source-DNA well (variant). For each variant we:

1. **Background-subtract**: subtract the mean negative-control rate from each well's raw rate.
2. **Concentration-normalize**: divide by mScarlett concentration to get activity per unit enzyme. This corrects for differences in expression level between wells.
3. **Aggregate replicates**: compute mean, std, and CV of the normalized rate across replicates.
4. **Filter**:
   - Require at least `mscarlett_num_pass_threshold` replicates above the mScarlett threshold (confirms the enzyme was expressed)
   - Require CV below `cv_threshold` (confirms replicates are consistent)

In [ ]:
kinetics_fit_agg_list = []

for round_number in kinetics_fit_df["round_number"].unique():
    tmp_round = kinetics_fit_df[kinetics_fit_df["round_number"] == round_number]
    for sample in tmp_round["sample_number"].unique():
        tmp_sample = tmp_round[tmp_round["sample_number"] == sample]
        for plate in tmp_sample["plate_number"].unique():
            tmp = tmp_sample[tmp_sample["plate_number"] == plate]

            bg_rate = tmp[
                (tmp["sample_type"] == "negative_control") & (tmp["r2_score"] > 0.9)
            ]["raw_rate"].mean()

            for well in tmp["source_dna_well"].unique():
                well_data = tmp[tmp["source_dna_well"] == well].copy()

                n_pass = (well_data["mscarlett_um_mean"] >= mscarlett_threshold).sum()
                well_data["mscarlett_num_pass"] = n_pass

                if n_pass > 0:
                    well_data = well_data[well_data["mscarlett_um_mean"] >= mscarlett_threshold].copy()

                well_data["bg_sub_rate"] = well_data["raw_rate"] - bg_rate
                well_data["bg_sub_conc_norm_rate"] = (
                    well_data["bg_sub_rate"] / (well_data["mscarlett_um_mean"] + 1e-4)
                )
                well_data["bg_sub_conc_norm_rate_mean"] = well_data["bg_sub_conc_norm_rate"].mean()

                if len(well_data) == 1:
                    well_data["bg_sub_conc_norm_rate_std"] = well_data["bg_sub_conc_norm_rate"].mean()
                else:
                    well_data["bg_sub_conc_norm_rate_std"] = well_data["bg_sub_conc_norm_rate"].std()

                well_data["bg_sub_conc_norm_rate_cv"] = (
                    well_data["bg_sub_conc_norm_rate_std"]
                    / (well_data["bg_sub_conc_norm_rate_mean"] + 1e-4)
                )

                to_drop = ["raw_rate", "bg_sub_rate", "bg_sub_conc_norm_rate", "well"]
                kinetics_fit_agg_list.append(well_data.drop(columns=to_drop).head(1))

kinetics_fit_agg_df = pd.concat(kinetics_fit_agg_list).reset_index(drop=True)
print(f"Aggregated variants (before filtering): {len(kinetics_fit_agg_df)}")

In [ ]:
filtered_data = kinetics_fit_agg_df[
    (kinetics_fit_agg_df["mscarlett_num_pass"] >= mscarlett_num_pass_threshold)
    & (kinetics_fit_agg_df["bg_sub_conc_norm_rate_cv"] <= cv_threshold)
]

print(f"After filtering: {len(filtered_data)} variants (removed {len(kinetics_fit_agg_df) - len(filtered_data)})")
print(f"  mScarlett pass filter removed: {(kinetics_fit_agg_df['mscarlett_num_pass'] < mscarlett_num_pass_threshold).sum()}")
print(f"  CV filter removed: {(kinetics_fit_agg_df['bg_sub_conc_norm_rate_cv'] > cv_threshold).sum()}")

## Step 4: Normalize to Positive Control

To compare across plates and rounds, we normalize each variant's activity to the **positive control** on that plate. This gives us a "fold-over-reference" metric:
- A value of 1.0 means the variant has the same activity as the reference (G20 PETase in this case)
- A value of 2.0 means twice the reference activity

This normalization is critical because absolute rates can vary between plates due to differences in substrate concentration, temperature, or instrument calibration.

In [ ]:
filtered_agg_list = []

for round_number in filtered_data["round_number"].unique():
    tmp_round = filtered_data[filtered_data["round_number"] == round_number]
    for sample in tmp_round["sample_number"].unique():
        tmp_sample = tmp_round[tmp_round["sample_number"] == sample]
        for plate in tmp_sample["plate_number"].unique():
            tmp = tmp_sample[tmp_sample["plate_number"] == plate].copy()

            wt_mean = tmp[tmp["sample_type"] == "positive_control"][
                "bg_sub_conc_norm_rate_mean"
            ].mean()

            tmp["g20_norm_rate"] = tmp["bg_sub_conc_norm_rate_mean"] / wt_mean
            filtered_agg_list.append(tmp)

filtered_data = pd.concat(filtered_agg_list).reset_index(drop=True)

print(f"Positive control mean rate: {wt_mean:.6f}")
print(f"\nNormalized activity distribution:")
print(filtered_data["g20_norm_rate"].describe().to_string())

## Step 5: Visualize Results

A few key plots to inspect the processed data before using it for model training.

In [ ]:
sample_data = filtered_data[filtered_data["sample_type"] == "sample"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5), dpi=100)

# Activity distribution
axes[0].hist(sample_data["g20_norm_rate"], bins=30, color="steelblue", alpha=0.7, edgecolor="white")
axes[0].axvline(1.0, color="red", linestyle="--", linewidth=1, label="G20 reference")
axes[0].set_xlabel("Activity (fold over G20)")
axes[0].set_ylabel("Count")
axes[0].set_title("Activity Distribution")
axes[0].legend()

# Expression vs. activity
axes[1].scatter(
    sample_data["mscarlett_um_mean"],
    sample_data["g20_norm_rate"],
    alpha=0.4, s=20,
)
axes[1].set_xlabel("mScarlett (µM)")
axes[1].set_ylabel("Activity (fold over G20)")
axes[1].set_title("Expression vs. Activity")

plt.tight_layout()
plt.show()

n_above_ref = (sample_data["g20_norm_rate"] > 1.0).sum()
print(f"{n_above_ref} / {len(sample_data)} variants ({n_above_ref/len(sample_data):.0%}) exceed the G20 reference")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), dpi=100)

# R² distribution
axes[0].hist(kinetics_fit_df["r2_score"], bins=30, color="steelblue", alpha=0.7, edgecolor="white")
axes[0].set_xlabel("R² Score")
axes[0].set_ylabel("Count")
axes[0].set_title("Kinetics Fit Quality (all wells)")

# CV distribution (pre-filter)
cv_vals = kinetics_fit_agg_df["bg_sub_conc_norm_rate_cv"].dropna()
axes[1].hist(cv_vals.clip(upper=5), bins=30, color="steelblue", alpha=0.7, edgecolor="white")
axes[1].axvline(cv_threshold, color="red", linestyle="--", linewidth=1, label=f"CV threshold = {cv_threshold}")
axes[1].set_xlabel("Coefficient of Variation")
axes[1].set_ylabel("Count")
axes[1].set_title("Replicate CV Distribution (pre-filter)")
axes[1].legend()

plt.tight_layout()
plt.show()

## Step 6: Save Processed Data

The final filtered DataFrame is saved as a CSV. This is the input format expected by the surrogate model training pipeline (`cleo-optimize-train`).

Key columns in the output:
- **`sequence`** — amino acid sequence of the variant
- **`name`** — variant identifier (encodes fragment composition)
- **`g20_norm_rate`** — normalized activity (fold over positive control)
- **`bg_sub_conc_norm_rate_mean`** — absolute background-subtracted, expression-normalized rate
- **`mscarlett_um_mean`** — average mScarlett concentration (expression proxy)
- **`r2_score`** — kinetics fit quality
- **`mscarlett_num_pass`** — number of replicates passing the expression threshold
- **`bg_sub_conc_norm_rate_cv`** — replicate coefficient of variation

In [ ]:
# Preview the final data
display_cols = [
    "name", "sequence", "sample_type", "round_number",
    "g20_norm_rate", "bg_sub_conc_norm_rate_mean",
    "mscarlett_um_mean", "r2_score",
    "mscarlett_num_pass", "bg_sub_conc_norm_rate_cv",
]
filtered_data[display_cols].head(10)

In [ ]:
# Uncomment to save (the example output is already provided)
# output_path = EXAMPLE_DIR / "output" / "processed_filtered_data.csv"
# filtered_data.to_csv(output_path, index=False)
# print(f"Saved {len(filtered_data)} rows to {output_path}")

## Summary

### Key decisions in data processing

| Decision | Why it matters |
|---|---|
| **mScarlett threshold** | Below-threshold wells likely have no enzyme; including them adds noise. Too high a threshold discards real data. |
| **CV threshold** | High-CV variants have unreliable measurements. But some real biological variability is expected — don't set too low. |
| **Background subtraction** | Essential to remove non-enzymatic 4-MU signal. Use negative controls from the *same plate* to account for plate-to-plate variation. |
| **Expression normalization** | Without this, high-expressing variants look more active even if their per-enzyme rate is the same. mScarlett co-expression corrects for this. |
| **Positive-control normalization** | Allows comparison across plates and rounds by converting to fold-over-reference. |
| **Which time points to use** | t0 may have mixing artifacts; very late time points may show substrate depletion. Pick the linear range. |

### What to check before training a model

- **R² scores**: Most wells should have R² > 0.9. If many are low, the kinetics may be non-linear (change time range or check assay).
- **Negative controls**: Should have near-zero rates. Elevated background suggests contamination.
- **Positive controls**: Should be consistent across plates. High variability here means plate-level normalization may not fully correct.
- **CV distribution**: If most variants have high CV, consider increasing replicate count or improving the assay.
- **Expression vs. activity**: Should show no strong correlation after normalization. A residual correlation means the normalization isn't fully correcting for expression differences.